# Cell 1 – Install all required libraries

In [1]:
# Install sentence-transformers for embedding models
!pip install -q sentence-transformers

# Install FAISS CPU version for vector search
!pip install -q faiss-cpu

# Install PyPDF2 to read text from PDF files
!pip install -q PyPDF2

# Install llama-cpp-python to run local GGUF LLM models
!pip install -q llama-cpp-python

# Install Gradio for optional web UI
!pip install -q gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 14.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00


# Cell 2 – Upload your PDF notes (ai_notespdf.pdf) & extract text

In [2]:
# Import Google Colab's file upload utility
from google.colab import files

# Open a file chooser so you can upload your PDF (ai_notespdf.pdf)
uploaded = files.upload()

# Get the first (and usually only) uploaded filename
pdf_filename = list(uploaded.keys())[0]

# Import PyPDF2 to read PDF content
import PyPDF2

# Define a function that extracts all text from a given PDF file
def extract_text_from_pdf(pdf_path):
    # Open the PDF file in read-binary mode
    with open(pdf_path, "rb") as f:
        # Create a PDF reader object
        reader = PyPDF2.PdfReader(f)
        # Create an empty string to store all pages' text
        full_text = ""
        # Loop through every page in the PDF
        for page in reader.pages:
            # Extract text from the current page and add to full_text
            full_text += page.extract_text() + "\n"
    # Return the combined text from all pages
    return full_text

# Call the function to get text from your uploaded PDF
raw_text = extract_text_from_pdf(pdf_filename)

# Print a small preview to confirm extraction worked
print("✅ Extracted text preview:")
print(raw_text[:500])


Saving ai_notespdf.pdf to ai_notespdf.pdf
✅ Extracted text preview:
Artificial Intelligence (AI) has rapidly become one of the most transformative technologies of the 21st century. It refers to computer systems designed to perform tasks that normall
y require human intelligence, such as learning, reasoning, problem-solving, speech recognition, and decision-making. AI systems are created using algorithms and vast amounts of data
, allowing them to recognize patterns and improve over time. These systems enable machines to understand language, detect emotions, driv


# Cell 3 – Chunk the text (150–300 words per chunk)

In [3]:
# Import typing for type hints (optional but good practice)
from typing import List

# Define a function to split long text into smaller word-based chunks
def chunk_text(text: str, chunk_size: int = 250) -> List[str]:
    # Split the text into a list of individual words
    words = text.split()
    # Create an empty list to store all chunks
    chunks = []
    # Loop over the words list, step = chunk_size
    for i in range(0, len(words), chunk_size):
        # Join a slice of words into one chunk string
        chunk = " ".join(words[i:i + chunk_size])
        # Add the chunk string to the chunks list
        chunks.append(chunk)
    # Return the list of all chunks
    return chunks

# Call the chunking function using the extracted text
chunks = chunk_text(raw_text, chunk_size=250)

# Print how many chunks were created
print(f"✅ Total chunks created: {len(chunks)}")

# Print a sample chunk to check
print("\nSample chunk:\n")
print(chunks[0])


✅ Total chunks created: 1

Sample chunk:

Artificial Intelligence (AI) has rapidly become one of the most transformative technologies of the 21st century. It refers to computer systems designed to perform tasks that normall y require human intelligence, such as learning, reasoning, problem-solving, speech recognition, and decision-making. AI systems are created using algorithms and vast amounts of data , allowing them to recognize patterns and improve over time. These systems enable machines to understand language, detect emotions, drive vehicles, diagnose diseases, and even creat e artwork. Machine Learning (ML), a subset of AI, focuses on training models with data so that they can make predictions or decisions without being explicitly programmed. Popular examples incl ude recommendation systems used by platforms like Netflix and YouTube, fraud detection in banking, and personalized ads on social media. Another advanced subfield is Deep Learning, which uses neural networks inspired by

# Cell 4 – Generate embeddings for each chunk

In [4]:
# Import the SentenceTransformer class to create embeddings
from sentence_transformers import SentenceTransformer

# Load a small, free embedding model (as suggested in assignment)
embedding_model_name = "all-MiniLM-L6-v2"

# Create an instance of the embedding model
embedder = SentenceTransformer(embedding_model_name)

# Generate embeddings (vectors) for every chunk
chunk_embeddings = embedder.encode(chunks)

# Print the shape of the embeddings (num_chunks, embedding_dimension)
print("✅ Embeddings shape:", chunk_embeddings.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embeddings shape: (1, 384)


# Cell 5 – Build FAISS vector store

In [5]:
# Import faiss for similarity search
import faiss
# Import numpy for numerical array handling
import numpy as np

# Get the dimension (length) of one embedding vector
embedding_dimension = chunk_embeddings.shape[1]

# Create a FAISS index using L2 (Euclidean) distance
index = faiss.IndexFlatL2(embedding_dimension)

# Convert embeddings to a numpy array (if not already)
emb_array = np.array(chunk_embeddings).astype("float32")

# Add all chunk embeddings to the FAISS index
index.add(emb_array)

# Print how many vectors are stored
print("✅ Number of vectors in FAISS index:", index.ntotal)


✅ Number of vectors in FAISS index: 1


# Cell 6 – Function to retrieve relevant chunks for a question

In [6]:
# Define a function to retrieve top-k similar chunks given a query
def retrieve_relevant_chunks(query: str, k: int = 3):
    # Convert the query text into an embedding vector
    query_embedding = embedder.encode([query]).astype("float32")
    # Use FAISS to search for the k nearest neighbors (similar vectors)
    distances, indices = index.search(query_embedding, k)
    # Extract the corresponding text chunks using the returned indices
    retrieved_chunks = [chunks[i] for i in indices[0]]
    # Return both the chunks and distances (optional)
    return retrieved_chunks, distances[0]

# Test the retrieval with a sample question
test_query = "What is Artificial Intelligence?"
retrieved, dists = retrieve_relevant_chunks(test_query, k=2)

# Print retrieved chunks to verify
print("✅ Retrieved chunks for query:", test_query)
for i, (chunk, dist) in enumerate(zip(retrieved, dists)):
    print(f"\n--- Chunk {i+1} (distance={dist:.4f}) ---\n")
    print(chunk)


✅ Retrieved chunks for query: What is Artificial Intelligence?

--- Chunk 1 (distance=0.8787) ---

Artificial Intelligence (AI) has rapidly become one of the most transformative technologies of the 21st century. It refers to computer systems designed to perform tasks that normall y require human intelligence, such as learning, reasoning, problem-solving, speech recognition, and decision-making. AI systems are created using algorithms and vast amounts of data , allowing them to recognize patterns and improve over time. These systems enable machines to understand language, detect emotions, drive vehicles, diagnose diseases, and even creat e artwork. Machine Learning (ML), a subset of AI, focuses on training models with data so that they can make predictions or decisions without being explicitly programmed. Popular examples incl ude recommendation systems used by platforms like Netflix and YouTube, fraud detection in banking, and personalized ads on social media. Another advanced subfield

# Cell 7 – Load a small LLM (GGUF) with llama-cpp-python

In [7]:
!pip install -q sentence-transformers faiss-cpu PyPDF2 llama-cpp-python gradio huggingface-hub


In [8]:
!pip install -q huggingface_hub

from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
    filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"
)

print("Model downloaded to:", model_path)


tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf:   0%|          | 0.00/669M [00:00<?, ?B/s]

Model downloaded to: /root/.cache/huggingface/hub/models--TheBloke--TinyLlama-1.1B-Chat-v1.0-GGUF/snapshots/52e7645ba7c309695bec7ac98f4f005b139cf465/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf


In [9]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=2048,       # Small model --> smaller context
    temperature=0.3,
    top_p=0.95,
    verbose=False
)

print("🚀 Model Loaded Successfully!")


🚀 Model Loaded Successfully!


# Cell 8 – RAG Answer Function (uses your LLM + FAISS)

In [10]:

def rag_answer(query: str, k: int = 3) -> str:
    """
    Use Retrieval-Augmented Generation (RAG) to answer a question.
    1) Retrieve top-k relevant chunks from your notes.
    2) Send them + question to the LLM.
    3) Return the final answer text.
    """

    # 1️ Retrieve relevant chunks from the FAISS index
    retrieved_chunks, _ = retrieve_relevant_chunks(query, k=k)

    # Join the retrieved chunks into a single context string
    context = "\n\n".join(retrieved_chunks)

    # 2️ Build a clear instruction prompt for the model
    prompt = f"""You are a helpful assistant for a course-notes question-answering task.
Use ONLY the following context to answer the question.
If the answer is not in the context, say "I am not sure based on the notes."

Context:
{context}

Question: {query}
Answer:"""

    # 3️ Call the LLM with the prompt to generate an answer
    output = llm(
        prompt,         # The text we send to the model
        max_tokens=256  # Limit on how long the answer can be
    )

    # 4️ Extract the generated text portion from the model's response
    answer = output["choices"][0]["text"]

    # Clean leading/trailing spaces/newlines and return
    return answer.strip()


#  Quick test of the full RAG pipeline
test_q = "What is Artificial Intelligence according to the notes?"
print(" Question:", test_q)
print(" Answer:\n", rag_answer(test_q))


 Question: What is Artificial Intelligence according to the notes?
 Answer:
 Artificial Intelligence (AI) is a subset of AI, which focuses on training models with data so that they can make predictions or decision without being explicitly programmed.


# Cell 9 – Optional Gradio UI (Ask questions interactively)

In [11]:

import gradio as gr  # Import Gradio library

def gradio_rag_interface(user_question: str) -> str:
    """
    Wrapper function for Gradio.
    Takes a user question from the UI and returns the RAG answer.
    """
    # Call our RAG answer function with the question
    return rag_answer(user_question, k=3)

# Create a simple web UI with one input box and one output box
demo = gr.Interface(
    fn=gradio_rag_interface,                  # Function to call when user submits
    inputs=gr.Textbox(lines=2, label="Ask a question from your notes"),  # Question box
    outputs=gr.Textbox(label="Answer"),       # Answer display box
    title=" Course Notes RAG Bot",          # App title
    description="Ask questions based on the uploaded course notes PDF."  # Subtitle
)

# Launch the app; share=True also gives a public link
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f2c7da4518b4efce23.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
